In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

GOLDEN_DIR = Path("../data/golden")
REPORT_DIR = Path("../reports")

accounts = pd.read_csv(GOLDEN_DIR / "accounts_golden.csv")
payments = pd.read_csv(GOLDEN_DIR / "payments_golden.csv")
calls = pd.read_csv(GOLDEN_DIR / "calls_golden.csv")
targeting = pd.read_csv(GOLDEN_DIR / "daily_targeting_golden.csv")

payments["event_at"] = pd.to_datetime(
    payments["event_at"],
    errors="coerce"
)

calls["event_at"] = pd.to_datetime(
    calls["event_at"],
    errors="coerce"
)

targeting["target_date"] = pd.to_datetime(
    targeting["target_date"],
    errors="coerce"
)

In [5]:
recovery = payments[
    payments["payment_status"] == "SUCCESS"
].copy()

recovery["month"] = (
    recovery["event_at"]
    .dt.to_period("M")
    .astype(str)
)

monthly = (
    recovery
    .groupby("month")
    .agg(
        recovery=("amount", "sum"),
        recovered_accounts=("account_id", "nunique")
    )
    .reset_index()
)

monthly["recovery_per_account"] = (
    monthly["recovery"]
    / monthly["recovered_accounts"]
)

display(monthly)

,month,recovery,recovered_accounts,recovery_per_account
0,2026-01,1.872291e+08,2374,78866.523837
1,2026-02,1.702796e+08,2173,78361.534731
2,2026-03,1.891903e+08,2419,78210.139140
3,2026-04,1.752289e+08,2304,76054.223164
4,2026-05,1.843355e+08,2344,78641.417944
5,2026-06,1.758534e+08,2286,76926.268530
6,2026-07,1.872478e+08,2335,80191.792719
7,2026-08,4.710970e+07,608,77483.051497


In [6]:
print(targeting.columns.tolist())
display(targeting.head())

['target_id', 'account_id', 'campaign_id', 'target_date', 'priority', 'recommended_channel', 'status']


,target_id,account_id,campaign_id,target_date,priority,recommended_channel,status
0,TGT0000001,ACC0028555,CMP0000103,2026-04-22,4,FIELD,QUEUED
1,TGT0000002,ACC0007194,CMP0000100,2026-08-06,10,SMS,EXPIRED
2,TGT0000003,ACC0029550,CMP0000074,2026-05-20,5,SMS,CONTACTED
3,TGT0000004,ACC0012329,CMP0000046,2026-07-30,10,WHATSAPP,QUEUED
4,TGT0000005,ACC0018387,CMP0000118,2026-07-10,7,WHATSAPP,EXPIRED


In [7]:
targeting["month"] = (
    targeting["target_date"]
    .dt.to_period("M")
    .astype(str)
)

targeting["period"] = np.where(
    targeting["month"] <= "2026-03",
    "pre_change",
    "post_change"
)

display(
    targeting[
        ["month", "period"]
    ].drop_duplicates().sort_values("month")
)

,month,period
17,2026-01,pre_change
7,2026-02,pre_change
5,2026-03,pre_change
0,2026-04,post_change
2,2026-05,post_change
8,2026-06,post_change
3,2026-07,post_change
1,2026-08,post_change


In [8]:
targeting_summary = (
    targeting
    .groupby("period")
    .agg(
        targeted_accounts=("account_id", "nunique"),
        targeting_events=("target_id", "nunique")
    )
    .reset_index()
)

display(targeting_summary)

,period,targeted_accounts,targeting_events
0,post_change,17640,26632
1,pre_change,13720,18368


In [9]:
channel_shift = pd.crosstab(
    targeting["period"],
    targeting["recommended_channel"],
    normalize="index"
) * 100

display(channel_shift.round(2))

recommended_channel,FIELD,SMS,VOICE,WHATSAPP
period,,,,
post_change,25.27,24.85,25.18,24.70
pre_change,25.23,25.06,24.48,25.23


In [10]:
pre_recovery = monthly[
    monthly["month"].isin(
        ["2026-01", "2026-02", "2026-03"]
    )
]["recovery_per_account"].mean()

print(
    f"Pre-change recovery per recovered account: "
    f"₹{pre_recovery:,.2f}"
)

Pre-change recovery per recovered account: ₹78,479.40


In [11]:
targeted_monthly = (
    targeting
    .groupby("month")["account_id"]
    .nunique()
    .reset_index(
        name="targeted_accounts"
    )
)

counterfactual = monthly.merge(
    targeted_monthly,
    on="month",
    how="left"
)

counterfactual[
    "recovery_per_targeted_account"
] = (
    counterfactual["recovery"]
    / counterfactual["targeted_accounts"]
)

display(counterfactual)

,month,recovery,recovered_accounts,recovery_per_account,targeted_accounts,recovery_per_targeted_account
0,2026-01,1.872291e+08,2374,78866.523837,5732,32663.839426
1,2026-02,1.702796e+08,2173,78361.534731,5160,32999.925382
2,2026-03,1.891903e+08,2419,78210.139140,5666,33390.456509
3,2026-04,1.752289e+08,2304,76054.223164,5585,31374.920353
4,2026-05,1.843355e+08,2344,78641.417944,5800,31781.979941
5,2026-06,1.758534e+08,2286,76926.268530,5535,31771.174320
6,2026-07,1.872478e+08,2335,80191.792719,5666,33047.623720
7,2026-08,4.710970e+07,608,77483.051497,1566,30082.819483


In [12]:
pre_efficiency = (
    counterfactual[
        counterfactual["month"].isin(
            ["2026-01", "2026-02", "2026-03"]
        )
    ]["recovery_per_targeted_account"]
    .mean()
)

print(
    f"Pre-change recovery per targeted account: "
    f"₹{pre_efficiency:,.2f}"
)

Pre-change recovery per targeted account: ₹33,018.07


In [13]:
post = counterfactual[
    counterfactual["month"].isin(
        [
            "2026-04",
            "2026-05",
            "2026-06",
            "2026-07"
        ]
    )
].copy()

post["counterfactual_recovery"] = (
    post["targeted_accounts"]
    * pre_efficiency
)

post["incremental_recovery"] = (
    post["recovery"]
    - post["counterfactual_recovery"]
)

display(
    post[
        [
            "month",
            "targeted_accounts",
            "recovery",
            "counterfactual_recovery",
            "incremental_recovery"
        ]
    ]
)

,month,targeted_accounts,recovery,counterfactual_recovery,incremental_recovery
3,2026-04,5585,1.752289e+08,1.844059e+08,-9.177012e+06
4,2026-05,5800,1.843355e+08,1.915048e+08,-7.169344e+06
5,2026-06,5535,1.758534e+08,1.827550e+08,-6.901588e+06
6,2026-07,5666,1.872478e+08,1.870804e+08,1.674300e+05


In [14]:
actual_post_recovery = post["recovery"].sum()

counterfactual_recovery = (
    post["counterfactual_recovery"].sum()
)

incremental_recovery = (
    actual_post_recovery
    - counterfactual_recovery
)

print(
    f"Actual post-change recovery: "
    f"₹{actual_post_recovery:,.2f}"
)

print(
    f"Counterfactual recovery: "
    f"₹{counterfactual_recovery:,.2f}"
)

print(
    f"Estimated incremental recovery: "
    f"₹{incremental_recovery:,.2f}"
)

Actual post-change recovery: ₹722,665,699.69
Counterfactual recovery: ₹745,746,214.22
Estimated incremental recovery: ₹-23,080,514.53


In [16]:
channel_targeting = (
    targeting
    .groupby("recommended_channel")
    .agg(
        targeted_accounts=("account_id", "nunique"),
        targeting_events=("target_id", "nunique")
    )
    .reset_index()
)

channel_accounts = targeting[
    ["account_id", "recommended_channel"]
].drop_duplicates()

channel_performance = channel_accounts.merge(
    recovery[["account_id", "amount"]],
    on="account_id",
    how="left"
)

channel_performance["amount"] = (
    channel_performance["amount"].fillna(0)
)

channel_performance = (
    channel_performance
    .groupby("recommended_channel")
    .agg(
        targeted_accounts=("account_id", "nunique"),
        recovery=("amount", "sum")
    )
    .reset_index()
)

channel_performance["recovery_per_targeted_account"] = (
    channel_performance["recovery"]
    / channel_performance["targeted_accounts"]
)

display(
    channel_performance.sort_values(
        "recovery_per_targeted_account",
        ascending=False
    )
)

,recommended_channel,targeted_accounts,recovery,recovery_per_targeted_account
1,SMS,9360,4.171320e+08,44565.386750
0,FIELD,9493,4.158013e+08,43800.828131
2,VOICE,9393,4.103514e+08,43686.936045
3,WHATSAPP,9363,4.089854e+08,43681.016807


In [17]:
counterfactual.to_csv(
    REPORT_DIR / "counterfactual_analysis.csv",
    index=False
)

post.to_csv(
    REPORT_DIR / "counterfactual_post_change.csv",
    index=False
)

channel_performance.to_csv(
    REPORT_DIR / "channel_performance.csv",
    index=False
)

print("Phase 6 analysis saved.")

Phase 6 analysis saved.


In [18]:
investment = pd.DataFrame({
    "item": [
        "Actual post-change recovery",
        "Counterfactual recovery",
        "Estimated impact of targeting change",
        "Best observed channel",
        "Best channel recovery/account",
        "Recommended investment",
        "Expected incremental recovery",
        "Investment",
        "Expected ROI",
        "Confidence"
    ],
    "value": [
        actual_post_recovery,
        counterfactual_recovery,
        incremental_recovery,
        channel_performance.loc[
            channel_performance[
                "recovery_per_targeted_account"
            ].idxmax(),
            "recommended_channel"
        ],
        channel_performance[
            "recovery_per_targeted_account"
        ].max(),
        "Better borrower targeting",
        "Not reliably estimable from current data",
        100_000_000,
        "Not reliably estimable",
        "Low"
    ]
})

display(investment)

investment.to_csv(
    REPORT_DIR / "investment_recommendation.csv",
    index=False
)

,item,value
0,Actual post-change recovery,722665699.69
1,Counterfactual recovery,745746214.220515
2,Estimated impact of targeting change,-23080514.530515
3,Best observed channel,SMS
4,Best channel recovery/account,44565.38675
5,Recommended investment,Better borrower targeting
6,Expected incremental recovery,Not reliably estimable from current data
7,Investment,100000000
8,Expected ROI,Not reliably estimable
9,Confidence,Low
